# Deepfake Detection: Xception + Attention + FFT


In [ ]:


import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import timm

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

DATASET_PATH = r"I:\Research\Deepfake Detection program\dataset"

NUM_FRAMES = 15
IMG_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 50

print("CUDA Available:", torch.cuda.is_available())

class DeepfakeDataset(Dataset):

    def __init__(self, video_paths, labels):
        self.video_paths = video_paths
        self.labels = labels

    def extract_frames(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frames = []

        if total_frames <= 0:
            cap.release()
            return torch.zeros(NUM_FRAMES,3,IMG_SIZE,IMG_SIZE)

        frame_ids = np.linspace(0,max(total_frames-1,0),NUM_FRAMES,dtype=int)

        for frame_id in frame_ids:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
            success, frame = cap.read()

            if not success:
                frame = np.zeros((IMG_SIZE,IMG_SIZE,3),dtype=np.uint8)

            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame,(IMG_SIZE,IMG_SIZE))
            frame = frame.astype(np.float32)/255.0
            frame = torch.tensor(frame).permute(2,0,1)

            frames.append(frame)

        cap.release()
        return torch.stack(frames)

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        return self.extract_frames(self.video_paths[idx]), self.labels[idx]

video_paths = []
labels = []

for file in os.listdir(os.path.join(DATASET_PATH,"real")):
    if file.endswith((".mp4",".avi",".mov")):
        video_paths.append(os.path.join(DATASET_PATH,"real",file))
        labels.append(0)

for file in os.listdir(os.path.join(DATASET_PATH,"fake")):
    if file.endswith((".mp4",".avi",".mov")):
        video_paths.append(os.path.join(DATASET_PATH,"fake",file))
        labels.append(1)

train_paths, val_paths, train_labels, val_labels = train_test_split(
    video_paths, labels, test_size=0.2, random_state=42, stratify=labels
)

train_loader = DataLoader(
    DeepfakeDataset(train_paths, train_labels),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    DeepfakeDataset(val_paths, val_labels),
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Testing DataLoader...") #---

videos, labels = next(iter(train_loader))

print("Videos Shape:", videos.shape)
print("Labels Shape:", labels.shape)

class AttentionBlock(nn.Module):
    def __init__(self, feature_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(feature_dim,512),
            nn.Tanh(),
            nn.Linear(512,1)
        )

    def forward(self,x):
        scores = self.attention(x)
        weights = torch.softmax(scores,dim=1)
        return (weights*x).sum(dim=1)

class FFTBranch(nn.Module):
    def __init__(self,input_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim,512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512,256),
            nn.ReLU(),
            nn.Linear(256,128)
        )

    def forward(self,x):
        fft = torch.abs(torch.fft.fft(x,dim=-1))
        fft = fft.mean(dim=1)
        return self.fc(fft)

class DeepfakeXception(nn.Module):

    def __init__(self):
        super().__init__()

        self.backbone = timm.create_model(
            "legacy_xception",
            pretrained=True,
            num_classes=0
        )

        self.feature_dim = 2048

        self.attention = AttentionBlock(self.feature_dim)
        self.fft_branch = FFTBranch(self.feature_dim)

        self.fusion = nn.Sequential(
            nn.Linear(2176,512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512,256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.classifier = nn.Sequential(
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128,2)
        )

    def forward(self,x):
        B,T,C,H,W = x.shape

        x = x.view(B*T,C,H,W)
        features = self.backbone(x)
        features = features.view(B,T,self.feature_dim)

        attention_feature = self.attention(features)
        fft_feature = self.fft_branch(features)

        fused = torch.cat([attention_feature,fft_feature],dim=1)
        fused = self.fusion(fused)

        return self.classifier(fused)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

temp_model = DeepfakeXception().to(device)

videos = videos.to(device)

with torch.no_grad():#--

    outputs = temp_model(videos)

print("Output Shape:", outputs.shape)

del temp_model
torch.cuda.empty_cache()

model = DeepfakeXception().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)

best_val_acc = 0
best_train_acc = 0

best_train_epoch = 0
best_val_epoch = 0

for epoch in range(EPOCHS):

    # =========================
    # TRAINING
    # =========================

    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    for videos, labels in train_loader:

        videos = videos.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(videos)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

        preds = outputs.argmax(1)

        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_loss = train_loss / len(train_loader)

    train_acc = 100 * train_correct / train_total

    if train_acc > best_train_acc:

        best_train_acc = train_acc

        best_train_epoch = epoch + 1

    # =========================
    # VALIDATION
    # =========================

    model.eval()

    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for videos, labels in val_loader:

            videos = videos.to(device)
            labels = labels.to(device)

            outputs = model(videos)

            loss = criterion(outputs, labels)

            val_loss += loss.item()

            preds = outputs.argmax(1)

            val_correct += (preds == labels).sum().item()

            val_total += labels.size(0)

    val_loss = val_loss / len(val_loader)

    val_acc = 100 * val_correct / val_total

    scheduler.step(val_acc)

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        best_val_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            "best_xception_attention_fft.pth"
        )

        

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%"
    )

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)

print(
    f"Best Training Accuracy : "
    f"{best_train_acc:.2f}% "
    f"(Epoch {best_train_epoch})"
)

print(
    f"Best Validation Accuracy : "
    f"{best_val_acc:.2f}% "
    f"(Epoch {best_val_epoch})"
)

print(
    "Best Model File : best_xception_attention_fft.pth"
)


i:\Research\Deepfake Detection program\deepfake\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA Available: True
Testing DataLoader...
Videos Shape: torch.Size([8, 15, 3, 224, 224])
Labels Shape: torch.Size([8])
Output Shape: torch.Size([8, 2])
Epoch [1/50] | Train Loss: 0.7160 | Train Acc: 47.38% | Val Loss: 0.7006 | Val Acc: 44.50%
Epoch [2/50] | Train Loss: 0.7130 | Train Acc: 48.25% | Val Loss: 0.7025 | Val Acc: 45.00%
Epoch [3/50] | Train Loss: 0.7037 | Train Acc: 50.50% | Val Loss: 0.7039 | Val Acc: 42.75%
Epoch [4/50] | Train Loss: 0.7074 | Train Acc: 49.62% | Val Loss: 0.7021 | Val Acc: 43.25%
Epoch [5/50] | Train Loss: 0.6977 | Train Acc: 51.94% | Val Loss: 0.6948 | Val Acc: 51.75%
Epoch [6/50] | Train Loss: 0.6757 | Train Acc: 57.69% | Val Loss: 0.6401 | Val Acc: 65.75%
Epoch [7/50] | Train Loss: 0.5625 | Train Acc: 72.25% | Val Loss: 0.7339 | Val Acc: 55.75%
Epoch [8/50] | Train Loss: 0.4059 | Train Acc: 83.56% | Val Loss: 0.3912 | Val Acc: 79.25%
Epoch [9/50] | Train Loss: 0.2885 | Train Acc: 88.31% | Val Loss: 0.2964 | Val Acc: 84.75%
Epoch [10/50] | Train Loss: 